# 06 — Gate 2: the first 1 km pool run (frequency-ensemble spec v0.9.1 §7)

**Reference cell = S0, k = 50, g = 5%.** Triple duty: **pool-cost measurement** (pool time ÷
single-MILP time prices the 14-cell ensemble), the **decisive degeneracy test** (§2.9 — the
fail branch is live), and the **E4 seed**. Three runs into three DISTINCT folders (the clobber
guard compares only targets+weights, which are identical here):

| run | folder | config |
|---|---|---|
| LP twin | `iter9_y2y_s0_lp` | manifest defaults (HiGHS ipm, proportion) — M5.5: a twin beside every certified MILP |
| certified optimum | `iter9_y2y_s0_single` | Gurobi binary, opt_gap 1e-4, NumericFocus — T1's intended-vs-realized source |
| **the pool** | `iter9_y2y_s0_pool` | + `portfolio_n=50, portfolio_gap=0.05` (opt_gap stays 1e-4 — the anchor certificate) |

Runs 1–2 should be fast; run 3's cost is UNKNOWN — that is the measurement. Time limit 12 h;
fine to leave running (WLS licence needs **live internet** throughout). All cells resumable —
a run with a `run_summary.json` is skipped. Kernel: `R (y2y)`. After this: `07_gate2_analysis`.

In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))   # pr_* functions (incl. the Gurobi-13 pool shim)

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
ctx   <- pr_setup(mpath, PROJ)

manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y


In [2]:
# ---- Ingest + planning units ONCE (arm-independent; every run reuses this context) ----
ctx <- modifyList(ctx, pr_ingest(ctx))
ctx <- modifyList(ctx, pr_planning_units(ctx))

ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget


In [3]:
# ---- S0 from the Gate-1 freeze (spec/scenarios_v1.json) ------------------------------------
sc  <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/scenarios_v1.json"))
S0W <- sc$S0_balanced$weights    # 7 continuous multipliers (mean-1); EFGs + intactness stay at baseline
S0T <- sc$S0_balanced$targets    # {irrecoverable_carbon_m_soc: 0.332}
cat(sprintf("scenarios_v1: derived %s | split %s (frac_soc %.3f) | audit %s\n",
            sc$`_meta`$derived_utc, sc$`_meta`$split_rule,
            as.numeric(sc$`_meta`$frac_soc), sc$`_meta`$audit_created_utc))
cat("S0 targets:\n")
for (nm in names(S0T)) cat(sprintf("  %-32s %.3f\n", nm, as.numeric(S0T[[nm]])))
cat("S0 weight multipliers:\n")
for (nm in names(S0W)) cat(sprintf("  %-32s %.4f\n", nm, as.numeric(S0W[[nm]])))

scenarios_v1: derived 2026-08-27T18:49:16.825895+00:00 | split mass (frac_soc 0.742) | audit 2026-08-26T22:32:08.502910+00:00
S0 targets:
  irrecoverable_carbon_m_soc       0.332
S0 weight multipliers:
  climate_type_macrorefugia        1.4600
  transboundary_connectivity       0.6693
  climate_corridors                1.1714
  irrecoverable_carbon_m_soc       0.4646
  irrecoverable_carbon_biomass     0.1986
  aoh_richness_birds               1.3286
  aoh_richness_mammals             1.7075


In [4]:
# ---- runner: one S0 run per solver configuration (resumable; same chain as 02_solve BATCH) --
run_s0 <- function(ctx, subdir, ov = list()) {
  done <- file.path(PROJ, "output_data", subdir, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("== %s already solved -- skipped\n", subdir)); return(invisible(NULL)) }
  cat(sprintf("\n===================== %s =====================\n", subdir))
  # pr_override returns a modified COPY of ctx -- check the printed EFFECTIVE targets/weights line
  actx <- do.call(pr_override, c(list(ctx,
      targets                    = S0T,
      feature_weight_multipliers = S0W,
      results_subdir             = subdir), ov))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  cat(sprintf("== %s DONE: %d solution(s), %.0f s solve\n",
              subdir, sv$n_sol, sv$timing[["elapsed"]]))
  invisible(NULL)
}

In [5]:
# ---- run 1/3: the LP twin (HiGHS ipm, proportion -- manifest defaults) ----------------------
run_s0(ctx, "iter9_y2y_s0_lp")


===================== iter9_y2y_s0_lp =====================
  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464639, irrecoverable_carbon_biomass=0.198559, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  override results_subdir   -> iter9_y2y_s0_lp
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.332 | weight multipliers: climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464639, irrecoverable_carbon_biomass=0.198559, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  outputs  -> output_data/iter9_y2y_s0_lp
weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)
  up-weight climate_type_macrorefugia x1.5 -> 1.4600
  up-weight transboundary_connectivity x0.7 -> 0.6693
  up-weight climate_corridor

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 1.707509)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 43200, …)
# ℹ Use `summary(...)` to see complete formulation.



LP has 49 rows; 1272962 cols; 16912559 nonzeros

Coefficient ranges:

  Matrix  [1e-04, 1e+05]

  Cost    [3e-02, 2e+00]

  Bound   [1e+00, 1e+00]

  RHS     [3e+04, 4e+05]


Presolving model

49 rows, 1081933 cols, 14251642 nonzeros 3s

2 rows, 965366 cols, 1930491 nonzeros 6357s

Presolve reductions: rows 2(-47); columns 965366(-307596); nonzeros 1930491(-14982068) 

Solving the presolved LP

IPX model has 2 rows, 965366 columns and 1930491 nonzeros

Input
    Number of variables:                                965366
    Number of free variables:                           0
    Number of constraints:                              2
    Number of equality constraints:                     0
    Number of matrix entries:                           1930491

    Matrix range:                                       [1e-04, 3e+04]

    RHS range:                                          [2e+04, 2e+05]

    Objective range:                                    [3e-06, 5e-01]

    Bounds range:  

In [6]:
# ---- run 2/3: the certified S0 optimum (binary MILP, Gurobi, opt_gap 1e-4) ------------------
run_s0(ctx, "iter9_y2y_s0_single",
       ov = list(solver = "gurobi", decision_type = "binary",
                 opt_gap = 1e-4, portfolio_n = 1))


===================== iter9_y2y_s0_single =====================
  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464639, irrecoverable_carbon_biomass=0.198559, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  override results_subdir   -> iter9_y2y_s0_single
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 1
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.332 | weight multipliers: climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464639, irrecoverable_carbon_biomass=0.198559, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  outputs  -> output_data/iter9_y2y_s0_single
weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 1.707509)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xf533ca41
Model has 48 linear objective coefficients
Variable types: 48 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 2e+00]
  Bounds range     [1e+00, 1e+0

In [7]:
# ---- run 3/3: THE POOL (k=50, g=5%) -- the Gate-2 measurement. Cost unknown; leave running --
# opt_gap 1e-4 <= portfolio_gap 0.05 (prioritizr asserts this ordering). PoolSearchMode=2:
# Gurobi keeps exploring after the optimum to collect the 50 best solutions within 5%.
run_s0(ctx, "iter9_y2y_s0_pool",
       ov = list(solver = "gurobi", decision_type = "binary",
                 opt_gap = 1e-4, portfolio_n = 50, portfolio_gap = 0.05))


===================== iter9_y2y_s0_pool =====================
  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464639, irrecoverable_carbon_biomass=0.198559, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  override results_subdir   -> iter9_y2y_s0_pool
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 50
  override portfolio_gap    -> 0.05
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.332 | weight multipliers: climate_type_macrorefugia=1.46002, transboundary_connectivity=0.669317, climate_corridors=1.17139, irrecoverable_carbon_m_soc=0.464639, irrecoverable_carbon_biomass=0.198559, aoh_richness_birds=1.32857, aoh_richness_mammals=1.70751
  outputs  -> output_data/iter9_y2y_s0_pool
weights: 8 conti

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 1.707509)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   gap portfolio (`number_solutions` = 50, `pool_gap` = 0.05)
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
Set parameter PoolSolutions to value 50
Set parameter PoolSearchMode to value 2
Set parameter PoolGap to value 0.05
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10
PoolSolutions  50
PoolSearchMode  2
PoolGap  0.05

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xf533ca41
Model has 48 linear objective coefficients
Variable types: 48 co

In [8]:
# ---- pool-cost report (reads the three run_summary.json) ------------------------------------
runs <- c(lp = "iter9_y2y_s0_lp", single = "iter9_y2y_s0_single", pool = "iter9_y2y_s0_pool")
S <- lapply(runs, function(r) jsonlite::read_json(file.path(PROJ, "output_data", r, "run_summary.json")))
for (k in names(S))
  cat(sprintf("%-7s %-22s %8.1f s   %2d solution(s)   solver=%s\n",
              k, runs[[k]], S[[k]]$solve_seconds, S[[k]]$n_alternatives, S[[k]]$params$solver))
ratio <- S$pool$solve_seconds / S$single$solve_seconds
cat(sprintf("\npool overhead: %.1fx the single MILP\n", ratio))
cat(sprintf("14-cell ensemble projection: 14 x pool = %.1f h (+ LP twins ~%.0f min, + 14 single anchors ~%.0f min)\n",
            14 * S$pool$solve_seconds / 3600, 14 * S$lp$solve_seconds / 60,
            14 * S$single$solve_seconds / 60))
op <- S$pool$solver_provenance$objective
if (!is.null(op)) {
  op <- as.numeric(unlist(op))
  cat(sprintf("pool objectives: best %.6f, worst %.6f, span %.3f%% of best (pool gap 5%%)\n",
              min(op), max(op), 100 * (max(op) - min(op)) / min(op)))
}
cat("\nnext: analyses/y2y/07_gate2_analysis.ipynb (kernel y2y-geo)\n")

lp      iter9_y2y_s0_lp          6515.6 s    1 solution(s)   solver=highs
single  iter9_y2y_s0_single        54.8 s    1 solution(s)   solver=gurobi
pool    iter9_y2y_s0_pool        1799.0 s   50 solution(s)   solver=gurobi

pool overhead: 32.8x the single MILP
14-cell ensemble projection: 14 x pool = 7.0 h (+ LP twins ~1520 min, + 14 single anchors ~13 min)
pool objectives: best 5.362800, worst 5.362800, span 0.000% of best (pool gap 5%)

next: analyses/y2y/07_gate2_analysis.ipynb (kernel y2y-geo)


## Next

`07_gate2_analysis.ipynb` computes pool integrity checks, the **pre-registered degeneracy
verdict**, T1 intended-vs-realized (incl. the biomass headline + θ-tail capture rates), the
LP-twin tightness, and writes the E4 seed (`runs/gate2_s0_ref/`). The Gate-2 decision —
proceed to Gate 3 freeze vs pivot — is made with the chat on those numbers.